In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import glob
import os

# Load all CSVs
csv_files = sorted(glob.glob('converted_data_2022-10-17-00_2022-10-17-23/*.csv'))
df = pd.concat([pd.read_csv(f, low_memory=False) for f in csv_files], ignore_index=True)
print(f"Total rows: {len(df):,}")
print(f"Columns: {df.columns.tolist()}")

## Basic stats

In [ ]:
print(f"Unique users: {df['userid'].nunique():,}")
print(f"Unique tweets: {df['tweetid'].nunique():,}")
print(f"\nTweet types:")
print(df['tweet_type'].value_counts())
print(f"\nLanguages (top 10):")
print(df['lang'].value_counts().head(10))

## Potential label columns

In [ ]:
label_candidates = ['state', 'country', 'verified', 'lang', 'norm_country']
for col in label_candidates:
    if col in df.columns:
        n_unique = df[col].nunique()
        n_labeled = df[col].notna().sum()
        pct = 100 * n_labeled / len(df)
        print(f"{col:20s}  unique={n_unique:4d}  labeled={n_labeled:,} ({pct:.1f}%)")

## State distribution

In [ ]:
# Per-user (not per-tweet)
user_df = df.groupby('userid').agg(
    state=('state', 'first'),
    verified=('verified', 'first'),
    followers=('followers_count', 'first'),
    friends=('friends_count', 'first'),
    statuses=('statuses_count', 'first'),
    tweet_count=('tweetid', 'count'),
).reset_index()
print(f"Total users: {len(user_df):,}")
print(f"Users with state: {user_df['state'].notna().sum():,} ({100*user_df['state'].notna().mean():.1f}%)")

state_counts = user_df['state'].value_counts()
print(f"\nStates: {len(state_counts)}")
print(state_counts)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
state_counts.plot(kind='bar', ax=ax)
ax.set_title('Users per state')
ax.set_xlabel('State')
ax.set_ylabel('Users')
plt.tight_layout()
plt.show()

## Verified users

In [ ]:
verified = user_df['verified'].value_counts()
print(verified)
print(f"\nVerified rate: {100 * (user_df['verified'] == True).mean():.2f}%")

## Follower count distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
user_df['followers'].clip(upper=user_df['followers'].quantile(0.99)).hist(bins=50, ax=axes[0])
axes[0].set_title('Follower count (clipped at 99th pct)')
np.log1p(user_df['followers']).hist(bins=50, ax=axes[1])
axes[1].set_title('log(followers + 1)')
plt.tight_layout()
plt.show()

print(user_df['followers'].describe())

## Candidate labels: which ones have enough class balance?

In [ ]:
# Influencer tier (log follower buckets) - could be a good transfer label
user_df['follower_tier'] = pd.qcut(np.log1p(user_df['followers']), q=5, labels=['nano','micro','mid','macro','mega'])
print("Follower tier distribution:")
print(user_df['follower_tier'].value_counts())

# Activity tier
user_df['activity_tier'] = pd.qcut(user_df['tweet_count'], q=3, labels=['low','medium','high'])
print("\nActivity tier distribution:")
print(user_df['activity_tier'].value_counts())

## Sentiment distribution by state (does state correlate with sentiment?)

In [ ]:
if 'sent_vader' in df.columns:
    state_sentiment = df[df['state'].notna()].groupby('state')['sent_vader'].mean().sort_values()
    fig, ax = plt.subplots(figsize=(14, 4))
    state_sentiment.plot(kind='bar', ax=ax)
    ax.set_title('Mean VADER sentiment by state')
    ax.set_ylabel('Sentiment')
    plt.tight_layout()
    plt.show()

## Graph stats (from graph_data.pt)

In [ ]:
import torch
g = torch.load('../graph/graph_data.pt', map_location='cpu')
print("Keys:", list(g.keys()))
print(f"Nodes: {g['x'].shape[0]:,}  Features: {g['x'].shape[1]}")
print(f"Edges: {g['edge_index'].shape[1]:,}")
labeled = (g['y'] >= 0).sum().item()
print(f"Labeled nodes: {labeled:,} ({100*labeled/g['x'].shape[0]:.1f}%)")
print(f"Label names: {g['label_names']}")

y = g['y']
label_counts = [(g['label_names'][i], (y == i).sum().item()) for i in range(len(g['label_names']))]
label_counts.sort(key=lambda x: -x[1])
print(f"\nNodes per state (top/bottom 5):")
for name, count in label_counts[:5]:
    print(f"  {name}: {count:,}")
print("  ...")
for name, count in label_counts[-5:]:
    print(f"  {name}: {count:,}")

## Node degree distribution

In [ ]:
edge_index = g['edge_index']
degrees = torch.bincount(edge_index[0], minlength=g['x'].shape[0])
degrees_np = degrees.numpy()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(degrees_np.clip(max=int(np.percentile(degrees_np, 99))), bins=50)
axes[0].set_title('Out-degree (clipped at 99th pct)')
axes[1].hist(np.log1p(degrees_np), bins=50)
axes[1].set_title('log(out-degree + 1)')
plt.tight_layout()
plt.show()

print(f"Mean degree: {degrees_np.mean():.2f}")
print(f"Median degree: {np.median(degrees_np):.2f}")
print(f"Max degree: {degrees_np.max()}")
print(f"Isolated nodes (degree=0): {(degrees_np == 0).sum():,}")